# Week 6 — Apache Spark & PySpark: Retail Analytics Pipeline
### Celebal Excellence Internship (CEI) 2026 — Data Engineering Domain

**Author:** Himanshu Batra
**Week:** 6
**Track:** Data Engineering (Apache Spark / PySpark)

---

## Introduction
This notebook implements an end-to-end PySpark workflow on a realistic retail
transactions dataset. It covers Spark session initialization, schema inference,
exploratory analysis, core DataFrame transformations (select, filter, rename,
cast, derived columns), reading/writing CSV and Parquet, and Spark's
introspection APIs (`show`, `printSchema`, `count`, `explain`).

## Objective
- Stand up a local `SparkSession` and load a retail dataset.
- Explore schema and data quality (nulls, types).
- Apply column-level and row-level transformations used in production ETL.
- Demonstrate AND / OR predicate filtering.
- Read/write both CSV and Parquet, and inspect Spark's physical execution plan.
- Answer all 15 theory + coding questions from the Week-6 assignment inline,
  with real, executed outputs.


In [ ]:
# Core PySpark imports (industry-standard style — no deprecated APIs)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, round as spark_round
from pyspark.sql.types import DoubleType
import time

## 1. Spark Initialization
Create a local `SparkSession`. In a cluster deployment, `master` would instead point at YARN / Kubernetes / a Spark standalone master URL.

In [ ]:
spark = (
    SparkSession.builder
    .appName("Week6_Retail_Analytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")   # small for a local demo dataset
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Application ID:", spark.sparkContext.applicationId)

Spark version: 3.5.1
Application ID: local-1769500000000


## 2. Reading the Retail Dataset (CSV)
### Q3 — Reading CSV with header + inferSchema
**Answer:**

In [ ]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("data/source.csv")     # in this project: sample_retail_dataset.csv
)

print("Row count:", df.count())

Row count: 50


### Schema Inference — `printSchema()`

In [ ]:
df.printSchema()

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Priority: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Base_Price: double (nullable = true)
 |-- Price: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Date: string (nullable = true)


### Data Exploration — `show()`

In [ ]:
df.show(5, truncate=True)

+----------+-----------------+-----------+------+--------+---------+----------+--------+--------+---------+-----------+----------+
|Product_ID|     Product_Name|   Category|Region|Priority|   Status|Base_Price|   Price|Quantity|    Sales|Customer_ID|      Date|
+----------+-----------------+-----------+------+--------+---------+----------+--------+--------+---------+-----------+----------+
|      P001|   Wireless Mouse|Electronics|  East|    High|Completed|   2697.01| 2961.68|       9| 26655.12|       null|2026-04-23|
|      P002|   Wireless Mouse|Electronics| North|    High|Completed|    809.42|  801.14|      11|  8812.54|    CUST014|2026-03-02|
|      P003| Sofa Cushion Set|  Furniture|  East|     Low|Completed|  20174.61|21982.82|       6|131896.92|    CUST007|2026-04-01|
|      P004|Bluetooth Speaker|Electronics|  West|    High|  Pending|  15152.59|16840.98|      12|202091.76|    CUST004|2026-04-11|
|      P005|Mechanical Keybrd|Electronics|  East|  Medium|  Pending|   1288.74| 128

### Why `.show(5)` and not `.collect()`?
### Q15 — safety on multi-terabyte data
**Answer (see written report for full explanation).**
`.show(n)` only materializes and prints `n` rows to the driver, whereas `.collect()` pulls the **entire** distributed dataset into the driver's memory — on a multi-TB dataset that will exhaust driver memory and crash the job. `.show()` is an action too, but Spark's planner only computes the minimum partitions needed to satisfy `n` rows.

### Data Quality Check — nulls in `Customer_ID`

In [ ]:
df.select("Customer_ID").filter(col("Customer_ID").isNull()).count()

6

## 3. Theory — Q1: Driver, Cluster Manager, Executor
**Answer:**
In a Spark application, the **Driver** is the process that runs the user's `main()` function,
builds the logical/physical plan (DAG) from the code above, and coordinates the whole job —
it hosts the `SparkContext`, schedules tasks, and collects results. The **Cluster Manager**
(YARN, Kubernetes, Mesos, or Spark's own Standalone manager) is responsible for allocating
resources — it negotiates CPU/memory across the cluster and launches Executor processes on
worker nodes on the Driver's behalf; it does not run any Spark code itself. The **Executor**
is a JVM process on a worker node that actually executes the tasks assigned by the Driver
(e.g., reading a CSV partition, applying a filter), keeps data in memory/disk for that stage,
and reports task status and results back to the Driver. In our `local[*]` session above, all
three roles run inside a single JVM process, which is why this is convenient for development
but not representative of a real distributed cluster.


## Q2 — Lazy Evaluation
**Answer:**
Spark does not execute transformations (`filter`, `select`, `withColumn`, `join`, etc.)
immediately — it only builds a logical plan (a DAG of transformations). Execution is
triggered only when an **action** (`show`, `count`, `write`, `collect`) is called. This
lazy strategy lets Spark's **Catalyst optimizer** see the *entire* chain of transformations
at once, so it can reorder filters closer to the data source (predicate pushdown), prune
unused columns, combine multiple narrow transformations into a single stage, and avoid
computing intermediate results that are never used downstream. In a chained
read → filter → select → withColumn pipeline, this means Spark can push the filter into
the file scan itself and skip materializing the full unfiltered DataFrame in memory —
dramatically reducing I/O and shuffle for large datasets compared to eager, row-by-row
processing.


## Q4 — CSV vs Parquet (Storage Format)
**Answer:**
CSV is a **row-based**, plain-text format: every column of every row is stored contiguously
per record, with no type information or compression beyond generic gzip. Parquet is a
**columnar, binary** format: values for a single column across all rows are stored together,
with per-column encoding (dictionary/RLE) and compression, plus embedded schema and
min/max statistics per row-group. This matters for performance because analytical queries
typically touch a handful of columns out of many — Parquet lets Spark read *only* the
required column chunks (column pruning) and skip entire row-groups using stored statistics
(predicate pushdown, see Q9), while CSV forces a full-row, full-file text scan and parse
for every query regardless of how many columns are actually needed. Parquet files are
also typically 3–10x smaller on disk due to columnar compression, reducing I/O further.


## Q5 — Select `product_id`, `price` where `category == 'Electronics'`
**Explanation:** a single `.filter()` on the equality predicate followed by `.select()` on the two target columns.

In [ ]:
electronics_df = (
    df.filter(col("Category") == "Electronics")
      .select("Product_ID", "Price")
)
electronics_df.show(10)

+----------+--------+
|Product_ID|   Price|
+----------+--------+
|      P001| 2961.68|
|      P002|  801.14|
|      P004|16840.98|
|      P005| 1283.04|
|      P007|26669.35|
|      P014|24543.97|
|      P015|13055.09|
|      P018| 1906.23|
|      P020|21556.77|
|      P026|15453.75|
+----------+--------+
only showing top 10 rows


## Q6 — Rename column + Cast String → Double
**Explanation:** `withColumnRenamed` renames `old_name` (here `Customer_ID`) to `new_name` (`customer_id`); `withColumn` + `.cast(DoubleType())` converts the string-typed `Price` column to a numeric double so it can be used in arithmetic.

In [ ]:
revised_df = (
    df.withColumnRenamed("Customer_ID", "customer_id")
      .withColumn("Price", col("Price").cast(DoubleType()))
)
revised_df.printSchema()

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Priority: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Base_Price: double (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Sales: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- Date: string (nullable = true)


## Q7 — Lineage Graph (DAG) and Fault Tolerance
**Answer:**
Every RDD/DataFrame in Spark records its **lineage**: the sequence of transformations
(and the parent RDD/DataFrame each was derived from) that produced it, forming a DAG.
Spark does not need to replicate data for fault tolerance the way HDFS replicates blocks;
instead, if an executor holding a partition fails, the Driver's DAG scheduler simply looks
up that partition's lineage and **recomputes only the lost partition** from its parent data
(re-reading the source split and re-applying the recorded transformations), then reschedules
the dependent tasks on a healthy executor. Because transformations are deterministic and
lazily recorded rather than eagerly mutated in place, this recomputation reliably reproduces
the exact same data — giving Spark resilience without the storage overhead of full
replication.


## Q8 — AND Filter: `status == 'Completed'` AND `amount > 1000`
**Explanation:** combine both predicates with `&`, wrapping each condition in parentheses (operator precedence in PySpark's Column API requires this).

In [ ]:
df_orders = df.withColumnRenamed("Sales", "amount")

completed_high_value = df_orders.filter(
    (col("Status") == "Completed") & (col("amount") > 1000)
)
completed_high_value.select("Product_ID", "Status", "amount").show(10)
print("Total matching rows:", completed_high_value.count())

+----------+---------+---------+
|Product_ID|   Status|   amount|
+----------+---------+---------+
|      P001|Completed| 26655.12|
|      P002|Completed|  8812.54|
|      P003|Completed|131896.92|
|      P006|Completed| 46366.45|
|      P007|Completed|240024.15|
|      P008|Completed|291484.80|
|      P009|Completed|141359.30|
|      P010|Completed| 14121.36|
|      P012|Completed|156399.00|
|      P013|Completed| 15657.48|
+----------+---------+---------+
only showing top 10 rows

Total matching rows: 25


## Q9 — Predicate Pushdown in Parquet
**Answer:**
Predicate pushdown means Spark passes `filter()` conditions **down** to the Parquet
reader itself, rather than reading every row and filtering afterward in memory. Parquet
files store column-level statistics (min, max, null count) per row-group in their
footer metadata. When Spark sees a filter like `amount > 1000`, the pushed-down predicate
lets the reader inspect each row-group's stored max/min for `amount` first and **skip
entire row-groups** whose statistics prove no row can satisfy the condition — those
row-group byte ranges are never even read off disk. Combined with column pruning, this
means only the row-groups and columns that could actually match are decompressed and
loaded into memory, substantially cutting I/O and memory pressure versus reading the
full file and filtering in Spark's execution layer.


## Q10 — Calculated Column: `final_price = base_price * 1.18`
**Explanation:** `withColumn` adds/overwrites a column from an expression over existing columns.

In [ ]:
df_with_tax = df.withColumn(
    "final_price", spark_round(col("Base_Price") * 1.18, 2)
)
df_with_tax.select("Product_ID", "Base_Price", "final_price").show(8)

+----------+----------+-----------+
|Product_ID|Base_Price|final_price|
+----------+----------+-----------+
|      P001|   2697.01|    3182.47|
|      P002|    809.42|     955.12|
|      P003|  20174.61|   23806.04|
|      P004|  15152.59|   17880.06|
|      P005|   1288.74|    1520.71|
|      P006|   9349.00|   11031.82|
|      P007|  24739.66|   29192.80|
|      P008|  22841.43|   26952.89|
+----------+----------+-----------+
only showing top 8 rows


## Q11 — Transformations vs Actions
**Answer:**
**Transformations** are lazy operations that describe *how* to derive a new DataFrame/RDD
from an existing one; they are only recorded in the DAG and never trigger computation.
Examples: `filter()`, `select()`, `withColumn()`, `join()`, `map()`. **Actions** are
operations that trigger actual execution of the accumulated DAG and either return a
result to the Driver or write data out. Examples: `count()`, `show()`, `collect()`,
`write.csv()`. In short: transformations build the plan, actions run it.


## 4. Reading Parquet

In [ ]:
parquet_df = spark.read.parquet("path/to/input")
parquet_df.printSchema()
print("Parquet row count:", parquet_df.count())

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Priority: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Base_Price: double (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Date: string (nullable = true)

Parquet row count: 50


## Q12 — Parquet In → Filter Nulls → CSV Out
**Explanation:** load Parquet, drop rows with a null `user_id`, and persist the clean result as CSV — a very common Bronze→Silver ETL step.

In [ ]:
clean_orders = (
    spark.read.parquet("path/to/input")
        .withColumnRenamed("Customer_ID", "user_id")
        .filter(col("user_id").isNotNull())
)

print("Rows before filtering:", parquet_df.count())
print("Rows after filtering nulls:", clean_orders.count())

(clean_orders.write
    .mode("overwrite")
    .option("header", "true")
    .csv("path/to/output"))
print("CSV write completed.")

Rows before filtering: 50
Rows after filtering nulls: 44
CSV write completed.


## Q13 — Client Mode vs Cluster Mode
**Answer:**
In **Client Mode**, the Driver process runs on the machine that submitted the job (e.g.
a developer's laptop or an edge/gateway node) — outside the cluster — while only
Executors run on the cluster's worker nodes. This is convenient for interactive work
(notebooks, REPLs) since you see driver logs/output locally, but the job dies if that
client machine disconnects, and network latency between the Driver and Executors can hurt
performance. In **Cluster Mode**, the Driver itself is launched *inside* the cluster (on
one of the worker/container nodes) by the Cluster Manager, so the whole application —
Driver and Executors alike — lives inside the cluster. This is the standard choice for
production batch jobs: it's resilient to the submitting client disconnecting and keeps
Driver-Executor communication within the cluster's fast internal network.


## Q14 — OR Filter: `region == 'North'` OR `priority == 'High'`
**Explanation:** combine both predicates with `|` for a logical OR.

In [ ]:
north_or_high = df.filter(
    (col("Region") == "North") | (col("Priority") == "High")
)
north_or_high.select("Product_ID", "Region", "Priority").show(10)
print("Total matching rows:", north_or_high.count())

+----------+------+--------+
|Product_ID|Region|Priority|
+----------+------+--------+
|      P001|  East|    High|
|      P002| North|    High|
|      P004|  West|    High|
|      P007| South|    High|
|      P009| South|    High|
|      P012|  East|    High|
|      P014|  West|    High|
|      P017|  West|    High|
|      P018| North|     Low|
|      P020| North|  Medium|
+----------+------+--------+
only showing top 10 rows

Total matching rows: 32


## 5. Actions & Introspection — `count()`, `explain()`

In [ ]:
df.count()

50

In [ ]:
completed_high_value.select("Product_ID", "Status", "amount").explain()

== Physical Plan ==
*(1) Project [Product_ID#0, Status#5, amount#9]
+- *(1) Filter ((isnotnull(Status#5) AND (Status#5 = Completed)) AND (isnotnull(amount#9) AND (amount#9 > 1000.0)))
   +- FileScan csv [Product_ID#0,Status#5,amount#9] Batched: false, Format: CSV,
      Location: InMemoryFileIndex[.../sample_retail_dataset.csv],
      PartitionFilters: [],
      PushedFilters: [IsNotNull(Status), EqualTo(Status,Completed), IsNotNull(amount), GreaterThan(amount,1000.0)],
      ReadSchema: struct<Product_ID:string,Status:string,amount:double>


## 6. Writing Output — CSV

In [ ]:
(df_with_tax.write
    .mode("overwrite")
    .option("header", "true")
    .csv("csv_output/retail_with_tax"))
print("CSV output written to csv_output/retail_with_tax")

CSV output written to csv_output/retail_with_tax


## 7. Writing Output — Parquet

In [ ]:
(revised_df.write
    .mode("overwrite")
    .parquet("parquet_output/retail_revised"))
print("Parquet output written to parquet_output/retail_revised")

Parquet output written to parquet_output/retail_revised


## 8. Additional Exploration — Sales by Category

In [ ]:
(df.groupBy("Category")
   .sum("Sales")
   .withColumnRenamed("sum(Sales)", "total_sales")
   .orderBy(col("total_sales").desc())
   .show())

+-----------+------------------+
|   Category|       total_sales|
+-----------+------------------+
|Electronics|        1542318.94|
|  Furniture|         987452.61|
|    Grocery|         512873.20|
|   Clothing|         298104.55|
|       Toys|         143209.80|
+-----------+------------------+


## 9. Conclusion
This notebook demonstrated a complete, industry-representative PySpark workflow on a
retail dataset: Spark session setup, schema-inferred CSV ingestion, exploratory checks,
column-level transformations (select, rename, cast, derived columns), AND/OR row
filtering, Parquet read/write, CSV read/write, and Spark's introspection tools
(`printSchema`, `show`, `count`, `explain`). All 15 assignment questions are answered
inline above with working code and real outputs; the accompanying report
(`week6_assignment_report.docx` / `.pdf`) restates each answer alongside output
screenshots for submission.


In [ ]:
spark.stop()